In [1]:
import numpy as np
import pandas as pd
from io import StringIO
from unlzw3 import unlzw
import arff
from scipy.io import arff as scipy_arff
from collections.abc import Generator
from scipy.io import loadmat
from typing import Dict
from scipy.sparse import lil_matrix, coo_matrix
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.3

# Colon Lung

In [3]:
data, meta = scipy_arff.loadarff("../Datasets/AP Colon Lung/AP_Colon_Lung.arff")
data = pd.DataFrame(data)
data.head(3)

,ID_REF,1007_s_at,117_at,121_at,1405_i_at,1438_at,1487_at,1494_f_at,1552256_a_at,1552257_a_at,...,AFFX-r2-Ec-bioC-3_at,AFFX-r2-Ec-bioC-5_at,AFFX-r2-Ec-bioD-3_at,AFFX-r2-Ec-bioD-5_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,AFFX-ThrX-3_at,AFFX-ThrX-5_at,AFFX-ThrX-M_at,Tissue
0,117656.0,3462.8,106.7,498.9,601.8,1330.3,1613.6,227.2,1708.7,1874.6,...,9516.2,9404.7,44719.9,37408.4,122346.8,94852.9,2029.9,991.7,1137.3,b'Colon'
1,301706.0,3138.9,234.7,708.4,447.3,1995.7,1692.6,177.3,3737.1,2462.3,...,6131.8,5243.2,24282.1,21226.7,67313.1,51688.1,2782.1,1356.7,1811.6,b'Colon'
2,203723.0,3614.6,367.9,885.4,602.1,823.4,1404.8,262.6,1660.8,1310.1,...,3681.1,3401.5,16639.9,14865.3,40291.2,31496.8,1034.3,405.1,612.6,b'Colon'


In [4]:
data["ID_REF"].nunique(), data.shape, data.isna().sum().sum()

(412, (412, 10937), np.int64(0))

In [5]:
data.drop(columns=["ID_REF"], inplace=True)

In [6]:
for col in data.select_dtypes([object]):
    # Tenta decodificar de bytes para string (UTF-8 é o padrão)
    data[col] = data[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

In [7]:
data.head(3)

,1007_s_at,117_at,121_at,1405_i_at,1438_at,1487_at,1494_f_at,1552256_a_at,1552257_a_at,1552281_at,...,AFFX-r2-Ec-bioC-3_at,AFFX-r2-Ec-bioC-5_at,AFFX-r2-Ec-bioD-3_at,AFFX-r2-Ec-bioD-5_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,AFFX-ThrX-3_at,AFFX-ThrX-5_at,AFFX-ThrX-M_at,Tissue
0,3462.8,106.7,498.9,601.8,1330.3,1613.6,227.2,1708.7,1874.6,758.4,...,9516.2,9404.7,44719.9,37408.4,122346.8,94852.9,2029.9,991.7,1137.3,Colon
1,3138.9,234.7,708.4,447.3,1995.7,1692.6,177.3,3737.1,2462.3,1467.1,...,6131.8,5243.2,24282.1,21226.7,67313.1,51688.1,2782.1,1356.7,1811.6,Colon
2,3614.6,367.9,885.4,602.1,823.4,1404.8,262.6,1660.8,1310.1,989.4,...,3681.1,3401.5,16639.9,14865.3,40291.2,31496.8,1034.3,405.1,612.6,Colon


In [8]:
data.rename(columns={"Tissue": "label"}, inplace=True)
data["label"].value_counts(normalize=True)

label
Colon    0.694175
Lung     0.305825
Name: proportion, dtype: float64

In [9]:
data.loc[data["label"] == "Colon", "label"] = 0
data.loc[data["label"] == "Lung", "label"] = 1
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.694175
1    0.305825
Name: proportion, dtype: float64

In [10]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,10926,10927,10928,10929,10930,10931,10932,10933,10934,label
0,3462.8,106.7,498.9,601.8,1330.3,1613.6,227.2,1708.7,1874.6,758.4,...,9516.2,9404.7,44719.9,37408.4,122346.8,94852.9,2029.9,991.7,1137.3,0
1,3138.9,234.7,708.4,447.3,1995.7,1692.6,177.3,3737.1,2462.3,1467.1,...,6131.8,5243.2,24282.1,21226.7,67313.1,51688.1,2782.1,1356.7,1811.6,0
2,3614.6,367.9,885.4,602.1,823.4,1404.8,262.6,1660.8,1310.1,989.4,...,3681.1,3401.5,16639.9,14865.3,40291.2,31496.8,1034.3,405.1,612.6,0


In [11]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [12]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,10927,10928,10929,10930,10931,10932,10933,10934,label,subset
0,3237.7,205.7,1019.7,229.5,1001.2,1225.9,417.9,1308.2,886.7,687.9,...,8089.6,36181.5,33828.5,134314.9,94486.8,3570.1,1470.7,2200.1,0,train
1,3315.3,309.9,952.1,183.4,1800.5,1592.9,194.5,1432.9,1523.7,742.1,...,5354.9,24378.7,21790.9,87330.4,64549.9,1007.5,360.1,659.4,0,train
2,3634.0,281.0,729.5,803.2,156.7,608.0,230.2,717.4,733.7,96.1,...,7438.4,34381.7,29437.2,131581.0,101056.2,2011.6,690.6,915.3,1,train
3,1680.6,575.7,888.9,647.8,702.9,510.4,234.2,892.9,1023.9,373.2,...,10513.4,46244.5,41558.8,129980.3,102039.8,1815.9,601.7,1149.0,1,train
4,3749.1,402.3,622.7,689.9,358.7,1125.2,199.1,633.6,2116.4,992.3,...,7379.0,35416.1,30002.4,103091.2,77087.6,1833.3,867.7,1066.2,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
407,3916.1,144.1,701.9,1107.2,1284.3,1663.9,239.7,1699.4,2072.8,1172.3,...,1144.0,18548.7,15430.2,50848.9,38966.9,911.1,362.8,511.9,0,test
408,2794.2,294.2,1030.9,146.5,2177.5,1515.7,248.4,1733.5,2108.5,492.7,...,5628.1,33141.0,27866.4,96371.7,72592.1,1792.0,878.8,928.1,0,test
409,1772.5,833.5,979.4,3618.1,42.9,538.2,245.0,1522.8,750.7,347.1,...,11425.3,61840.4,49970.5,143033.3,107638.8,2824.4,1826.5,2031.3,1,test
410,2969.9,300.6,532.2,1648.6,311.1,866.6,65.4,1023.8,855.3,549.8,...,1380.6,7957.4,7135.8,24231.2,18180.8,692.7,89.5,250.1,0,test


In [13]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.694175
 1    0.305825
 Name: proportion, dtype: float64,
 subset
 train    0.699029
 test     0.300971
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.693548
         1        0.306452
 train   0        0.694444
         1        0.305556
 Name: proportion, dtype: float64,
 np.int64(0))

In [14]:
data.to_parquet("../Benchmark/ap_colon_lung.parquet", index=False)

# Anthracycline Taxane Chemotherapy

In [15]:
data, meta = scipy_arff.loadarff("../Datasets/Anthracycline Taxane Chemotherapy/phpCLGrjq.arff")
data = pd.DataFrame(data)
print(data.shape)
data.head(3)

(159, 61360)


,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var61351,Var61352,Var61353,Var61354,Var61355,Var61356,Var61357,Var61358,Var61359,Decision
0,3.570,3.378,3.214,3.583,4.439,4.066,3.097,2.644,3.119,3.341,...,3.796,2.872,2.408,3.226,3.351,3.217,3.665,5.587,3.989,b'1'
1,3.751,3.481,3.020,3.386,3.381,4.105,3.117,2.609,3.012,3.509,...,3.861,2.786,2.431,2.968,3.355,3.278,3.660,5.833,4.042,b'2'
2,3.594,3.352,3.230,3.368,3.747,3.991,3.143,2.717,3.093,3.442,...,4.019,2.859,2.539,2.956,3.372,3.313,3.783,5.242,3.988,b'1'


In [16]:
for col in data.select_dtypes([object]):
    # Tenta decodificar de bytes para string (UTF-8 é o padrão)
    data[col] = data[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

In [17]:
data.head(3)

,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var61351,Var61352,Var61353,Var61354,Var61355,Var61356,Var61357,Var61358,Var61359,Decision
0,3.570,3.378,3.214,3.583,4.439,4.066,3.097,2.644,3.119,3.341,...,3.796,2.872,2.408,3.226,3.351,3.217,3.665,5.587,3.989,1
1,3.751,3.481,3.020,3.386,3.381,4.105,3.117,2.609,3.012,3.509,...,3.861,2.786,2.431,2.968,3.355,3.278,3.660,5.833,4.042,2
2,3.594,3.352,3.230,3.368,3.747,3.991,3.143,2.717,3.093,3.442,...,4.019,2.859,2.539,2.956,3.372,3.313,3.783,5.242,3.988,1


In [18]:
data["Decision"].value_counts(normalize=True)

Decision
1    0.597484
2    0.402516
Name: proportion, dtype: float64

In [19]:
data.rename(columns={"Decision": "label"}, inplace=True)
data.loc[data["label"] == "1", "label"] = 1
data.loc[data["label"] == "2", "label"] = 0
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
1    0.597484
0    0.402516
Name: proportion, dtype: float64

In [20]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,61350,61351,61352,61353,61354,61355,61356,61357,61358,label
0,3.570,3.378,3.214,3.583,4.439,4.066,3.097,2.644,3.119,3.341,...,3.796,2.872,2.408,3.226,3.351,3.217,3.665,5.587,3.989,1
1,3.751,3.481,3.020,3.386,3.381,4.105,3.117,2.609,3.012,3.509,...,3.861,2.786,2.431,2.968,3.355,3.278,3.660,5.833,4.042,0
2,3.594,3.352,3.230,3.368,3.747,3.991,3.143,2.717,3.093,3.442,...,4.019,2.859,2.539,2.956,3.372,3.313,3.783,5.242,3.988,1


In [21]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [22]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,61351,61352,61353,61354,61355,61356,61357,61358,label,subset
0,3.720,3.289,3.180,4.048,4.401,4.117,3.070,2.622,3.085,3.425,...,3.544,2.432,3.216,3.850,3.283,3.664,5.113,3.787,0,train
1,3.738,3.356,3.460,3.791,4.063,3.677,3.216,3.149,2.927,3.394,...,2.663,2.604,3.176,3.321,3.031,3.691,5.166,4.067,0,train
2,3.602,3.546,3.159,3.921,3.667,3.900,3.100,2.734,2.970,3.341,...,2.947,2.705,3.141,3.341,3.217,3.597,5.440,4.039,1,train
3,4.005,3.260,3.007,3.782,4.187,3.789,3.128,2.581,2.979,3.279,...,2.725,2.681,3.028,3.788,3.040,3.775,5.925,4.372,0,train
4,3.767,3.243,3.006,4.519,4.615,4.034,3.024,2.557,2.793,3.324,...,3.020,2.413,3.582,3.323,3.111,3.745,6.199,4.016,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,3.670,3.452,3.337,3.383,3.743,4.165,3.155,2.609,3.062,3.404,...,2.796,2.622,3.420,3.338,3.269,3.587,4.983,3.954,0,test
155,3.854,3.614,3.035,3.402,3.365,4.153,3.128,2.679,3.051,3.602,...,2.855,2.696,3.092,3.255,3.357,3.589,4.164,4.419,0,test
156,3.772,3.239,3.231,3.789,4.259,3.611,3.057,2.718,2.904,3.307,...,3.060,3.490,3.125,3.480,3.022,3.841,6.195,4.013,0,test
157,3.653,3.365,2.990,4.536,3.909,4.007,3.129,2.575,3.054,3.395,...,2.818,2.303,3.163,3.445,3.175,3.721,4.963,4.216,1,test


In [23]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 1    0.597484
 0    0.402516
 Name: proportion, dtype: float64,
 subset
 train    0.698113
 test     0.301887
 Name: proportion, dtype: float64,
 subset  label
 test    1        0.604167
         0        0.395833
 train   1        0.594595
         0        0.405405
 Name: proportion, dtype: float64,
 np.int64(0))

In [24]:
data.to_parquet("../Benchmark/anthracycline_taxane_chemotherapy.parquet", index=False)

# SMK-CAN-187

In [7]:
with open("../Datasets/SMK/dataset_", "r", encoding="utf-8") as f:
    decoder = arff.ArffDecoder()
    obj = decoder.decode(f.read(), encode_nominal=True)

# Extract metadata
attributes = obj["attributes"]
data = obj["data"]

# Build DataFrame
columns = [attr[0] for attr in attributes]
data = pd.DataFrame(data, columns=columns)

print(data.shape)
data.head()

(187, 19994)


,att2,att3,att4,att5,att6,att7,att8,att9,att10,att11,...,att19986,att19987,att19988,att19989,att19990,att19991,att19992,att19993,att19994,att1
0,10.697,5.345,7.142,4.814,6.948,6.704,8.101,7.175,7.753,6.138,...,5.783,8.114,7.373,5.467,5.103,5.500,5.209,7.569,6.748,1
1,10.424,5.463,7.246,5.098,5.946,6.795,8.260,7.395,7.823,5.915,...,6.532,8.254,7.514,5.759,5.337,5.732,4.215,8.010,6.859,1
2,10.374,5.482,7.815,4.935,6.463,6.772,8.410,7.510,8.013,6.273,...,6.503,8.594,8.052,5.968,5.619,6.601,4.249,8.144,6.928,1
3,10.500,5.584,7.404,5.140,5.764,6.999,8.433,7.568,7.591,5.830,...,6.571,8.304,7.576,5.366,5.491,5.776,4.582,7.526,6.966,1
4,9.820,5.612,7.511,5.263,5.562,6.713,8.312,7.464,7.704,5.550,...,6.761,8.693,8.326,6.438,5.280,5.735,4.619,8.983,7.247,1


In [8]:
data.isna().sum().sum()

np.int64(0)

In [10]:
data.dtypes, data["att1"].value_counts(normalize=True)

(att2        float64
 att3        float64
 att4        float64
 att5        float64
 att6        float64
              ...   
 att19991    float64
 att19992    float64
 att19993    float64
 att19994    float64
 att1         object
 Length: 19994, dtype: object,
 att1
 2    0.518717
 1    0.481283
 Name: proportion, dtype: float64)

In [11]:
data.rename(columns={"att1": "label"}, inplace=True)
data.loc[data["label"] == "1", "label"] = 1
data.loc[data["label"] == "2", "label"] = 0
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.518717
1    0.481283
Name: proportion, dtype: float64

In [12]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,19984,19985,19986,19987,19988,19989,19990,19991,19992,label
0,10.697,5.345,7.142,4.814,6.948,6.704,8.101,7.175,7.753,6.138,...,5.783,8.114,7.373,5.467,5.103,5.500,5.209,7.569,6.748,1
1,10.424,5.463,7.246,5.098,5.946,6.795,8.260,7.395,7.823,5.915,...,6.532,8.254,7.514,5.759,5.337,5.732,4.215,8.010,6.859,1
2,10.374,5.482,7.815,4.935,6.463,6.772,8.410,7.510,8.013,6.273,...,6.503,8.594,8.052,5.968,5.619,6.601,4.249,8.144,6.928,1


In [13]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [14]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,19985,19986,19987,19988,19989,19990,19991,19992,label,subset
0,9.638,5.844,7.207,5.434,5.592,6.949,8.419,7.692,7.313,5.676,...,8.603,8.369,6.137,4.961,5.201,5.183,8.619,7.124,0,train
1,9.615,5.573,7.047,5.547,5.780,7.145,8.422,7.318,7.459,5.758,...,8.408,7.965,5.704,5.167,5.964,5.077,8.117,7.303,0,train
2,10.134,5.549,7.073,5.083,5.932,6.779,8.737,7.485,7.505,5.769,...,8.519,7.739,5.098,5.398,5.657,4.533,7.404,6.928,0,train
3,10.561,5.788,7.667,5.254,5.243,7.101,8.491,7.540,8.079,5.734,...,8.353,7.715,5.598,5.453,5.670,4.375,8.201,7.133,0,train
4,10.467,5.786,7.589,5.233,4.745,6.512,8.456,7.595,7.748,5.955,...,8.298,7.898,5.409,5.446,5.612,4.369,8.122,7.201,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,9.291,5.863,7.324,5.948,4.854,7.040,8.781,7.828,7.027,5.218,...,8.624,7.898,5.519,4.936,5.882,4.599,7.774,7.528,0,test
183,10.285,5.542,7.648,5.113,6.268,6.723,8.887,7.359,7.578,6.324,...,8.798,7.927,5.468,5.605,6.119,4.401,7.468,6.968,0,test
184,10.697,5.345,7.142,4.814,6.948,6.704,8.101,7.175,7.753,6.138,...,8.114,7.373,5.467,5.103,5.500,5.209,7.569,6.748,1,test
185,10.084,5.613,7.488,5.153,5.568,7.032,8.478,7.513,7.503,6.614,...,8.750,8.133,5.366,5.542,5.552,4.984,7.638,7.228,1,test


In [15]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.518717
 1    0.481283
 Name: proportion, dtype: float64,
 subset
 train    0.695187
 test     0.304813
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.526316
         1        0.473684
 train   0        0.515385
         1        0.484615
 Name: proportion, dtype: float64,
 np.int64(0))

In [16]:
data.to_parquet("../Benchmark/smk_can.parquet", index=False)

# Gliomas

In [17]:
with open("../Datasets/Gliomas/dataset_", "r", encoding="utf-8") as f:
    decoder = arff.ArffDecoder()
    obj = decoder.decode(f.read(), encode_nominal=True)

# Extract metadata
attributes = obj["attributes"]
data = obj["data"]

# Build DataFrame
columns = [attr[0] for attr in attributes]
data = pd.DataFrame(data, columns=columns)

print(data.shape)
data.head()

(85, 22284)


,att2,att3,att4,att5,att6,att7,att8,att9,att10,att11,...,att22276,att22277,att22278,att22279,att22280,att22281,att22282,att22283,att22284,class
0,29844.273,1011.659,689.391,5198.424,269.881,2033.805,986.082,149.570,44.134,527.588,...,404.532,67839.32,53978.508,52.274,28.044,35.416,4.863,26.816,22.039,1
1,17265.617,793.230,584.338,4367.415,538.245,1342.165,1087.825,117.612,42.127,421.173,...,427.738,97757.89,80901.240,30.942,42.106,103.130,19.493,91.644,27.866,1
2,25947.941,880.637,862.301,4867.595,294.926,1036.071,979.293,93.095,28.982,494.550,...,496.755,84907.32,61828.957,77.635,33.896,16.005,4.074,37.106,9.872,1
3,29054.473,1000.799,842.576,6974.653,428.498,1890.961,1055.917,80.668,71.308,1010.702,...,617.696,114561.35,99823.070,79.836,58.045,54.872,11.043,48.565,30.257,1
4,30286.408,687.285,798.224,8020.007,404.549,2420.757,1398.942,248.498,67.267,945.524,...,1033.011,134436.27,106568.930,41.904,78.221,144.647,15.498,158.274,34.885,1


In [19]:
data["class"].value_counts(), data.isna().sum().sum()

(class
 2    59
 1    26
 Name: count, dtype: int64,
 np.int64(0))

In [20]:
data.rename(columns={"class": "label"}, inplace=True)
data.loc[data["label"] == "1", "label"] = 1
data.loc[data["label"] == "2", "label"] = 0
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.694118
1    0.305882
Name: proportion, dtype: float64

In [21]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,22274,22275,22276,22277,22278,22279,22280,22281,22282,label
0,29844.273,1011.659,689.391,5198.424,269.881,2033.805,986.082,149.570,44.134,527.588,...,404.532,67839.32,53978.508,52.274,28.044,35.416,4.863,26.816,22.039,1
1,17265.617,793.230,584.338,4367.415,538.245,1342.165,1087.825,117.612,42.127,421.173,...,427.738,97757.89,80901.240,30.942,42.106,103.130,19.493,91.644,27.866,1
2,25947.941,880.637,862.301,4867.595,294.926,1036.071,979.293,93.095,28.982,494.550,...,496.755,84907.32,61828.957,77.635,33.896,16.005,4.074,37.106,9.872,1


In [22]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,22275,22276,22277,22278,22279,22280,22281,22282,label,subset
0,30286.408,687.285,798.224,8020.007,404.549,2420.757,1398.942,248.498,67.267,945.524,...,134436.270,106568.930,41.904,78.221,144.647,15.498,158.274,34.885,1,train
1,14543.907,1991.069,3223.948,5649.142,63.952,2743.984,851.790,911.240,466.683,200.644,...,67405.970,61878.594,47.931,46.913,208.558,23.957,44.408,45.681,0,train
2,27075.715,237.787,414.418,6071.487,670.498,1723.033,460.246,62.021,100.764,286.953,...,115814.640,115779.360,66.722,65.477,243.287,178.151,25.199,24.611,0,train
3,37648.332,1874.908,1256.215,5620.608,247.378,3027.899,972.662,874.137,46.022,330.649,...,102951.920,83895.680,75.438,58.953,164.053,33.734,58.660,23.893,0,train
4,23500.504,735.818,575.802,5382.979,555.009,1109.377,872.735,69.671,68.935,131.558,...,127133.050,116329.330,92.674,56.084,154.120,32.588,62.135,40.287,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,30593.238,1128.039,809.379,6575.126,302.810,1555.777,1278.266,243.245,60.274,1086.237,...,129186.590,98356.125,46.904,39.120,64.008,103.905,65.122,30.967,1,test
81,20616.256,1009.929,735.139,3873.815,334.770,2274.018,991.010,124.071,45.250,568.368,...,97637.780,85499.550,79.598,105.811,173.766,37.663,65.420,34.438,0,test
82,15272.749,762.146,675.783,5122.911,629.184,1331.668,1027.185,47.108,101.810,305.262,...,80107.670,78652.055,58.712,152.592,36.003,91.267,31.213,27.187,0,test
83,28203.623,1219.907,1909.059,5276.717,166.480,2062.229,835.198,333.334,65.705,1037.946,...,42296.246,34281.426,36.561,27.397,158.146,59.504,29.170,22.847,0,test


In [23]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.694118
 1    0.305882
 Name: proportion, dtype: float64,
 subset
 train    0.694118
 test     0.305882
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.692308
         1        0.307692
 train   0        0.694915
         1        0.305085
 Name: proportion, dtype: float64,
 np.int64(0))

In [24]:
data.to_parquet("../Benchmark/gliomas.parquet", index=False)

# Breast

In [25]:
with open("../Datasets/Breast/dataset_", "r", encoding="utf-8") as f:
    decoder = arff.ArffDecoder()
    obj = decoder.decode(f.read(), encode_nominal=True)

# Extract metadata
attributes = obj["attributes"]
data = obj["data"]

# Build DataFrame
columns = [attr[0] for attr in attributes]
data = pd.DataFrame(data, columns=columns)

print(data.shape)
data.head()

(97, 24482)


,Contig45645_RC,Contig44916_RC,D25272,J00129,Contig29982_RC,Contig26811,D25274,Contig36292,Contig42854,Contig34839,...,NM_000898,NM_000899,Contig20164_RC,Contig8985_RC,Contig36062_RC,Contig35333_RC,Contig62037_RC,AF067420,Contig15167_RC,Class
0,-0.299,0.093,-0.215,-0.566,-0.596,-0.195,0.039,-0.409,-0.352,0.066,...,-0.960,-0.211,0.155,-0.095,-0.025,-0.037,0.215,0.307,0.321,relapse
1,-0.081,0.009,-0.091,-0.518,-0.502,-0.149,0.098,-0.090,0.138,0.061,...,-0.531,-0.020,0.014,-0.123,0.148,0.024,-0.070,-0.209,0.105,relapse
2,-0.125,0.070,-0.006,-0.575,-0.585,-0.183,0.102,0.023,-0.350,-0.005,...,-0.883,-0.159,0.022,0.006,-0.086,0.019,0.026,-0.822,0.199,relapse
3,-0.270,0.123,0.056,-0.499,-0.402,-0.099,-0.145,-0.103,0.181,0.236,...,-0.044,-0.096,0.018,0.000,0.076,0.057,-0.016,-0.360,-0.038,relapse
4,-0.141,0.025,-0.031,-0.465,-0.533,-0.065,0.101,-0.008,-0.019,0.026,...,0.280,-0.088,0.043,0.207,-0.124,-0.041,-0.077,-0.432,-0.015,relapse


In [26]:
data.isna().sum().sum()

np.int64(0)

In [27]:
data["Class"].value_counts(), data.isna().sum().sum()

(Class
 non-relapse    51
 relapse        46
 Name: count, dtype: int64,
 np.int64(0))

In [28]:
data.rename(columns={"Class": "label"}, inplace=True)
data.loc[data["label"] == "non-relapse", "label"] = 0
data.loc[data["label"] == "relapse", "label"] = 1
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.525773
1    0.474227
Name: proportion, dtype: float64

In [29]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,24472,24473,24474,24475,24476,24477,24478,24479,24480,label
0,-0.299,0.093,-0.215,-0.566,-0.596,-0.195,0.039,-0.409,-0.352,0.066,...,-0.960,-0.211,0.155,-0.095,-0.025,-0.037,0.215,0.307,0.321,1
1,-0.081,0.009,-0.091,-0.518,-0.502,-0.149,0.098,-0.090,0.138,0.061,...,-0.531,-0.020,0.014,-0.123,0.148,0.024,-0.070,-0.209,0.105,1
2,-0.125,0.070,-0.006,-0.575,-0.585,-0.183,0.102,0.023,-0.350,-0.005,...,-0.883,-0.159,0.022,0.006,-0.086,0.019,0.026,-0.822,0.199,1


In [30]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,24473,24474,24475,24476,24477,24478,24479,24480,label,subset
0,0.074,-0.047,-0.041,-0.509,0.459,-0.093,0.013,-0.178,-0.257,0.045,...,-0.013,-0.049,-0.040,0.050,-0.059,-0.073,-0.146,-0.028,1,train
1,-0.296,0.046,-0.229,1.551,-0.176,-0.340,-0.071,-0.144,-0.269,-0.157,...,0.090,0.211,0.052,0.102,0.089,0.301,-0.049,0.069,1,train
2,0.266,-0.202,-0.238,-0.545,0.227,-0.148,-0.008,-0.153,-0.064,-0.197,...,0.085,0.185,0.173,0.231,0.280,0.638,0.672,0.277,1,train
3,-0.231,0.078,0.099,-0.527,-0.478,-0.260,-0.205,-0.177,-0.200,-0.127,...,-0.106,0.134,0.340,-0.021,-0.011,0.145,-0.875,0.128,1,train
4,-0.276,0.004,0.012,0.894,-0.233,0.037,-0.138,-0.080,-0.454,-0.146,...,0.445,0.066,0.057,-0.078,0.140,0.353,-0.848,0.159,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,-0.382,0.064,0.033,-0.873,-0.474,0.148,-0.048,0.042,0.116,0.061,...,-0.227,-0.012,0.021,0.190,-0.006,-0.219,-0.073,-0.077,1,test
93,-0.343,0.302,0.169,-0.528,-0.447,0.283,0.032,0.167,-0.254,0.213,...,-0.020,-0.061,-0.160,-0.048,-0.056,0.022,0.140,-0.109,0,test
94,-0.279,0.054,-0.048,0.786,-0.164,-0.139,-0.021,0.052,-0.144,-0.047,...,-0.145,-0.015,-0.098,0.001,0.049,-0.077,0.219,-0.074,1,test
95,-0.212,-0.010,0.013,-0.511,-0.207,0.041,-0.010,0.130,0.175,0.021,...,-0.235,-0.056,0.001,0.047,-0.094,-0.149,0.170,-0.258,1,test


In [31]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.525773
 1    0.474227
 Name: proportion, dtype: float64,
 subset
 train    0.690722
 test     0.309278
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.533333
         1        0.466667
 train   0        0.522388
         1        0.477612
 Name: proportion, dtype: float64,
 np.int64(0))

In [32]:
data.to_parquet("../Benchmark/breast.parquet", index=False)

# Ovarian

In [33]:
with open("../Datasets/Ovarian/dataset_", "r", encoding="utf-8") as f:
    decoder = arff.ArffDecoder()
    obj = decoder.decode(f.read(), encode_nominal=True)

# Extract metadata
attributes = obj["attributes"]
data = obj["data"]

# Build DataFrame
columns = [attr[0] for attr in attributes]
data = pd.DataFrame(data, columns=columns)

print(data.shape)
data.head()

(253, 15155)


,MZ-7.86E-05,MZ2.18E-07,MZ9.60E-05,MZ0.000366014,MZ0.000810195,MZ0.001428564,MZ0.002221123,MZ0.003187869,MZ0.004328805,MZ0.005643929,...,MZ19974.404,MZ19977.042,MZ19979.68,MZ19982.319,MZ19984.957,MZ19987.596,MZ19990.235,MZ19992.874,MZ19995.513,Class
0,0.494626,0.263735,0.321841,0.220934,0.297622,0.316458,0.154763,0.223685,0.304346,0.241757,...,0.483622,0.449296,0.449296,0.449296,0.449296,0.449296,0.449296,0.449296,0.449296,Normal
1,0.258063,0.406593,0.321841,0.069771,0.333335,0.354432,0.321431,0.144740,0.260869,0.142853,...,0.631765,0.619718,0.619718,0.619718,0.619718,0.619718,0.619718,0.619718,0.619718,Normal
2,0.537636,0.032966,0.321841,0.209307,0.404762,0.113927,0.369049,0.223685,0.536231,0.131865,...,0.038462,0.035918,0.035918,0.035918,0.035918,0.035918,0.035918,0.035918,0.035918,Normal
3,0.000000,0.395605,0.310347,0.197673,0.404762,0.455701,0.416666,0.210527,0.420292,0.274723,...,0.497864,0.486621,0.486621,0.486621,0.486621,0.486621,0.486621,0.486621,0.486621,Normal
4,0.526884,0.395605,0.367817,0.383719,0.488099,0.392405,0.238094,0.500000,0.362316,0.274723,...,0.267096,0.251408,0.251408,0.251408,0.251408,0.251408,0.251408,0.251408,0.251408,Normal


In [34]:
data["Class"].value_counts(), data.isna().sum().sum()

(Class
 Cancer    162
 Normal     91
 Name: count, dtype: int64,
 np.int64(0))

In [36]:
data.rename(columns={"Class": "label"}, inplace=True)
data.loc[data["label"] == "Cancer", "label"] = 1
data.loc[data["label"] == "Normal", "label"] = 0
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
1    0.640316
0    0.359684
Name: proportion, dtype: float64

In [37]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,15145,15146,15147,15148,15149,15150,15151,15152,15153,label
0,0.494626,0.263735,0.321841,0.220934,0.297622,0.316458,0.154763,0.223685,0.304346,0.241757,...,0.483622,0.449296,0.449296,0.449296,0.449296,0.449296,0.449296,0.449296,0.449296,0
1,0.258063,0.406593,0.321841,0.069771,0.333335,0.354432,0.321431,0.144740,0.260869,0.142853,...,0.631765,0.619718,0.619718,0.619718,0.619718,0.619718,0.619718,0.619718,0.619718,0
2,0.537636,0.032966,0.321841,0.209307,0.404762,0.113927,0.369049,0.223685,0.536231,0.131865,...,0.038462,0.035918,0.035918,0.035918,0.035918,0.035918,0.035918,0.035918,0.035918,0


In [38]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,15146,15147,15148,15149,15150,15151,15152,15153,label,subset
0,0.430107,0.032966,0.298853,0.372092,0.321431,0.341774,0.321431,0.447370,0.449277,0.164836,...,0.638029,0.638029,0.638029,0.638029,0.638029,0.638029,0.638029,0.638029,1,train
1,0.666670,0.956045,0.804598,0.848837,0.666671,0.911395,0.690479,0.855267,0.927538,0.604395,...,0.516199,0.516199,0.516199,0.516199,0.516199,0.516199,0.516199,0.516199,1,train
2,0.344088,0.494503,0.482762,0.279072,0.249999,0.481016,0.511907,0.381582,0.565216,0.252746,...,0.238733,0.238733,0.238733,0.238733,0.238733,0.238733,0.238733,0.238733,1,train
3,0.311825,0.274723,0.310347,0.279072,0.452385,0.481016,0.178572,0.118418,0.173915,0.527475,...,0.680991,0.680991,0.680991,0.680991,0.680991,0.680991,0.680991,0.680991,0,train
4,0.462364,0.175819,0.218390,0.174418,0.226190,0.177216,0.309526,0.289473,0.304346,0.362638,...,0.143665,0.143665,0.143665,0.143665,0.143665,0.143665,0.143665,0.143665,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,0.645160,0.527469,0.632183,0.546510,0.428571,0.493674,0.642856,0.289473,0.492754,0.417582,...,0.359859,0.359859,0.359859,0.359859,0.359859,0.359859,0.359859,0.359859,1,test
249,0.301073,0.318678,0.494256,0.162791,0.214285,0.417721,0.440475,0.210527,0.231885,0.395605,...,0.523947,0.523947,0.523947,0.523947,0.523947,0.523947,0.523947,0.523947,0,test
250,0.236558,0.549452,0.517244,0.313954,0.452385,0.367090,0.345240,0.368424,0.594200,0.439559,...,0.400704,0.400704,0.400704,0.400704,0.400704,0.400704,0.400704,0.400704,1,test
251,0.849466,0.494503,0.356323,0.639536,0.809524,0.772153,0.583334,0.460527,0.884054,0.593407,...,0.432393,0.432393,0.432393,0.432393,0.432393,0.432393,0.432393,0.432393,1,test


In [39]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 1    0.640316
 0    0.359684
 Name: proportion, dtype: float64,
 subset
 train    0.699605
 test     0.300395
 Name: proportion, dtype: float64,
 subset  label
 test    1        0.644737
         0        0.355263
 train   1        0.638418
         0        0.361582
 Name: proportion, dtype: float64,
 np.int64(0))

In [40]:
data.to_parquet("../Benchmark/ovarian.parquet", index=False)

# PCam

In [42]:
data, meta = scipy_arff.loadarff("../Datasets/PCam/dataset.arff")
data = pd.DataFrame(data)
data.head(3)

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f27639,f27640,f27641,f27642,f27643,f27644,f27645,f27646,f27647,label
0,209.0,152.0,167.0,218.0,167.0,186.0,193.0,149.0,176.0,189.0,...,175.0,136.0,157.0,81.0,46.0,70.0,88.0,54.0,79.0,b'1'
1,111.0,71.0,167.0,80.0,36.0,131.0,99.0,52.0,146.0,131.0,...,188.0,127.0,207.0,182.0,119.0,200.0,139.0,75.0,159.0,b'1'
2,249.0,222.0,239.0,255.0,236.0,255.0,248.0,221.0,255.0,76.0,...,243.0,207.0,235.0,240.0,190.0,227.0,179.0,118.0,160.0,b'0'


In [43]:
for col in data.select_dtypes([object]):
    data[col] = data[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

In [44]:
data["label"].value_counts(normalize=True)

label
0    0.50575
1    0.49425
Name: proportion, dtype: float64

In [45]:
data.dtypes, data.isna().sum().sum()

(f0        float64
 f1        float64
 f2        float64
 f3        float64
 f4        float64
            ...   
 f27644    float64
 f27645    float64
 f27646    float64
 f27647    float64
 label      object
 Length: 27649, dtype: object,
 np.int64(0))

In [46]:
data["label"] = data["label"].astype(int)

In [47]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,27639,27640,27641,27642,27643,27644,27645,27646,27647,label
0,209.0,152.0,167.0,218.0,167.0,186.0,193.0,149.0,176.0,189.0,...,175.0,136.0,157.0,81.0,46.0,70.0,88.0,54.0,79.0,1
1,111.0,71.0,167.0,80.0,36.0,131.0,99.0,52.0,146.0,131.0,...,188.0,127.0,207.0,182.0,119.0,200.0,139.0,75.0,159.0,1
2,249.0,222.0,239.0,255.0,236.0,255.0,248.0,221.0,255.0,76.0,...,243.0,207.0,235.0,240.0,190.0,227.0,179.0,118.0,160.0,0


In [48]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,27640,27641,27642,27643,27644,27645,27646,27647,label,subset
0,200.0,120.0,189.0,141.0,72.0,129.0,209.0,152.0,193.0,234.0,...,81.0,163.0,94.0,66.0,153.0,68.0,35.0,126.0,0,train
1,90.0,45.0,88.0,90.0,39.0,82.0,76.0,19.0,60.0,123.0,...,51.0,95.0,90.0,39.0,95.0,107.0,58.0,123.0,1,train
2,223.0,169.0,203.0,200.0,151.0,181.0,204.0,161.0,188.0,184.0,...,242.0,251.0,255.0,243.0,253.0,208.0,182.0,193.0,0,train
3,247.0,240.0,232.0,235.0,228.0,220.0,217.0,208.0,203.0,211.0,...,108.0,152.0,167.0,122.0,165.0,161.0,116.0,157.0,1,train
4,216.0,177.0,222.0,73.0,32.0,76.0,103.0,61.0,101.0,107.0,...,248.0,243.0,244.0,243.0,241.0,255.0,250.0,252.0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,70.0,26.0,141.0,65.0,22.0,130.0,77.0,31.0,129.0,96.0,...,107.0,199.0,156.0,85.0,179.0,195.0,113.0,211.0,0,test
3996,243.0,210.0,241.0,211.0,177.0,210.0,184.0,150.0,185.0,246.0,...,69.0,166.0,101.0,40.0,143.0,140.0,73.0,178.0,1,test
3997,60.0,33.0,84.0,47.0,14.0,59.0,73.0,29.0,65.0,225.0,...,150.0,194.0,216.0,162.0,198.0,134.0,79.0,111.0,1,test
3998,235.0,137.0,222.0,228.0,135.0,216.0,243.0,157.0,232.0,234.0,...,133.0,210.0,186.0,94.0,177.0,186.0,97.0,181.0,0,test


In [49]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.50575
 1    0.49425
 Name: proportion, dtype: float64,
 subset
 train    0.7
 test     0.3
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.505833
         1        0.494167
 train   0        0.505714
         1        0.494286
 Name: proportion, dtype: float64,
 np.int64(0))

In [51]:
data.to_parquet("../Benchmark/pcam.parquet", index=False)

# Central Nervous System

In [57]:
with open("../Datasets/Central Nervous System/dataset_", "r", encoding="utf-8") as f:
    decoder = arff.ArffDecoder()
    obj = decoder.decode(f.read(), encode_nominal=True)

# Extract metadata
attributes = obj["attributes"]
data = obj["data"]

# Build DataFrame
columns = [attr[0] for attr in attributes]
data = pd.DataFrame(data, columns=columns)

print(data.shape)
data.head()

(60, 7130)


,AFFX-BioB-5_at,AFFX-BioB-M_at,AFFX-BioB-3_at,AFFX-BioC-5_at,AFFX-BioC-3_at,AFFX-BioDn-5_at,AFFX-BioDn-3_at,AFFX-CreX-5_at,AFFX-CreX-3_at,AFFX-BioB-5_st,...,U58516_at,U73738_at,X06956_at,X16699_at,X83863_at,Z17240_at,L49218_f_at,M71243_f_at,Z78285_f_at,CLASS
0,-60.0,-109.0,45.0,22.0,7.0,-61.0,-211.0,-102.0,-20.0,98.0,...,1082.0,-12.0,44.0,-48.0,123.0,89.0,-25.0,56.0,-63.0,1
1,-159.0,-113.0,-62.0,-16.0,-160.0,-395.0,97.0,-157.0,15.0,-8.0,...,1295.0,-677.0,-452.0,-750.0,1273.0,-1.0,-750.0,-45.0,-293.0,1
2,-119.0,-31.0,4.0,-11.0,-197.0,-541.0,-277.0,-166.0,17.0,392.0,...,592.0,55.0,88.0,-37.0,310.0,159.0,11.0,2.0,2.0,1
3,41.0,9.0,-256.0,370.0,-302.0,-680.0,254.0,31.0,186.0,358.0,...,433.0,71.0,-237.0,-166.0,52.0,-48.0,-183.0,328.0,-244.0,1
4,-165.0,-57.0,171.0,216.0,-692.0,-381.0,762.0,-6.0,44.0,216.0,...,2160.0,-120.0,-82.0,-25.0,3888.0,393.0,-38.0,190.0,38.0,1


In [58]:
data["CLASS"].value_counts(normalize=True), data.isna().sum().sum(), data["CLASS"].dtypes

(CLASS
 0    0.65
 1    0.35
 Name: proportion, dtype: float64,
 np.int64(0),
 dtype('O'))

In [59]:
data.rename(columns={"CLASS": "label"}, inplace=True)
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.65
1    0.35
Name: proportion, dtype: float64

In [60]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,7120,7121,7122,7123,7124,7125,7126,7127,7128,label
0,-60.0,-109.0,45.0,22.0,7.0,-61.0,-211.0,-102.0,-20.0,98.0,...,1082.0,-12.0,44.0,-48.0,123.0,89.0,-25.0,56.0,-63.0,1
1,-159.0,-113.0,-62.0,-16.0,-160.0,-395.0,97.0,-157.0,15.0,-8.0,...,1295.0,-677.0,-452.0,-750.0,1273.0,-1.0,-750.0,-45.0,-293.0,1
2,-119.0,-31.0,4.0,-11.0,-197.0,-541.0,-277.0,-166.0,17.0,392.0,...,592.0,55.0,88.0,-37.0,310.0,159.0,11.0,2.0,2.0,1


In [62]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,7121,7122,7123,7124,7125,7126,7127,7128,label,subset
0,-60.0,-109.0,45.0,22.0,7.0,-61.0,-211.0,-102.0,-20.0,98.0,...,-12.0,44.0,-48.0,123.0,89.0,-25.0,56.0,-63.0,1,train
1,-109.0,-80.0,-97.0,104.0,-77.0,-233.0,-32.0,-142.0,-103.0,263.0,...,-6.0,127.0,-11.0,330.0,0.0,18.0,35.0,-18.0,1,train
2,-120.0,80.0,106.0,77.0,-226.0,-234.0,338.0,-117.0,-58.0,162.0,...,202.0,66.0,-101.0,739.0,-21.0,-48.0,58.0,58.0,0,train


In [63]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.65
 1    0.35
 Name: proportion, dtype: float64,
 subset
 train    0.7
 test     0.3
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.666667
         1        0.333333
 train   0        0.642857
         1        0.357143
 Name: proportion, dtype: float64,
 np.int64(0))

In [64]:
data.to_parquet("../Benchmark/central_nervous_system.parquet", index=False)

# Lymphoma

In [65]:
data, meta = scipy_arff.loadarff("../Datasets/Lymphoma/lymphoma_2classes.arff")
data = pd.DataFrame(data)
data.head(3)

,GENE1835X,GENE1836X,GENE1865X,GENE1380X,GENE1933X,GENE1932X,GENE1931X,GENE1930X,GENE3129X,GENE3126X,...,GENE3931X,GENE2588X,GENE3120X,GENE6X,GENE5X,GENE3X,GENE2X,GENE48X,GENE47X,class
0,0.46,0.70,0.67,-0.23,0.00,0.09,-0.02,-0.57,-0.17,-0.25,...,0.40,0.02,0.79,0.64,0.16,1.22,1.37,-0.04,0.16,b'ACL'
1,0.02,0.59,0.45,0.55,-0.08,-0.15,-0.05,-0.38,-0.55,0.35,...,0.57,0.52,-0.23,0.30,0.09,-0.20,-0.05,-0.14,-1.15,b'ACL'
2,-0.32,-0.63,-0.46,-0.28,-0.96,-1.17,-1.13,-0.89,-0.49,-0.23,...,1.62,-0.01,NaN,0.29,-0.57,1.20,1.40,0.29,0.25,b'ACL'


In [66]:
for col in data.select_dtypes([object]):
    data[col] = data[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

In [67]:
data["class"].value_counts(normalize=True), data.isna().sum().sum()

(class
 ACL    0.511111
 GCL    0.488889
 Name: proportion, dtype: float64,
 np.int64(5948))

In [68]:
data.rename(columns={"class": "label"}, inplace=True)
data.loc[data["label"] == "ACL", "label"] = 1
data.loc[data["label"] == "GCL", "label"] = 0
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
1    0.511111
0    0.488889
Name: proportion, dtype: float64

In [69]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,4017,4018,4019,4020,4021,4022,4023,4024,4025,label
0,0.46,0.70,0.67,-0.23,0.00,0.09,-0.02,-0.57,-0.17,-0.25,...,0.40,0.02,0.79,0.64,0.16,1.22,1.37,-0.04,0.16,1
1,0.02,0.59,0.45,0.55,-0.08,-0.15,-0.05,-0.38,-0.55,0.35,...,0.57,0.52,-0.23,0.30,0.09,-0.20,-0.05,-0.14,-1.15,1
2,-0.32,-0.63,-0.46,-0.28,-0.96,-1.17,-1.13,-0.89,-0.49,-0.23,...,1.62,-0.01,NaN,0.29,-0.57,1.20,1.40,0.29,0.25,1


In [71]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data.tail(3)

,0,1,2,3,4,5,6,7,8,9,...,4018,4019,4020,4021,4022,4023,4024,4025,label,subset
42,0.06,0.46,0.14,-0.35,0.32,-0.28,-0.04,0.17,0.46,-0.29,...,0.39,0.60,-0.03,-0.22,-0.45,0.51,0.19,-0.50,1,test
43,0.07,0.26,0.53,0.22,-0.26,0.09,0.49,0.49,-0.66,-1.07,...,0.26,-0.47,-0.50,-0.84,-0.80,0.88,-0.02,-0.77,1,test
44,0.37,0.38,0.30,-0.08,-0.45,-0.22,0.34,-0.53,-0.26,-0.87,...,-0.25,-0.39,-0.33,0.13,0.49,0.75,0.46,0.80,0,test


In [72]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 1    0.511111
 0    0.488889
 Name: proportion, dtype: float64,
 subset
 train    0.688889
 test     0.311111
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.500000
         1        0.500000
 train   1        0.516129
         0        0.483871
 Name: proportion, dtype: float64,
 np.int64(5948))

In [73]:
data.to_parquet("../Benchmark/lymphoma.parquet", index=False)

# dbworld bodies stemmed

In [84]:
data, meta = scipy_arff.loadarff("../Datasets/dbworld bodies stemmed/phpEZ030X.arff")
data = pd.DataFrame(data)
for col in data.select_dtypes([object]):
    data[col] = data[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
data.head(3)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V3713,V3714,V3715,V3716,V3717,V3718,V3719,V3720,V3721,Class
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [85]:
data.dtypes.unique()

array([dtype('O')], dtype=object)

In [86]:
for col in data.columns:
    data[col] = data[col].astype(int)

In [87]:
data.dtypes.unique(), data["Class"].value_counts(), data.isna().sum().sum()

(array([dtype('int64')], dtype=object),
 Class
 1    35
 2    29
 Name: count, dtype: int64,
 np.int64(0))

In [88]:
data.rename(columns={"Class": "label"}, inplace=True)
data.loc[data["label"] == 2, "label"] = 0
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,3712,3713,3714,3715,3716,3717,3718,3719,3720,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [89]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,3713,3714,3715,3716,3717,3718,3719,3720,label,subset
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
60,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
61,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
62,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test


In [90]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64')], dtype=object),
 label
 1    0.546875
 0    0.453125
 Name: proportion, dtype: float64,
 subset
 train    0.6875
 test     0.3125
 Name: proportion, dtype: float64,
 subset  label
 test    1        0.550000
         0        0.450000
 train   1        0.545455
         0        0.454545
 Name: proportion, dtype: float64,
 np.int64(0))

In [91]:
data.to_parquet("../Benchmark/dbworld_bodies_stemmed.parquet", index=False)

# Binary Isolet

In [96]:
data, meta = scipy_arff.loadarff("../Datasets/Binary Isolet/file22f047870b9.arff")
data = pd.DataFrame(data)
for col in data.select_dtypes([object]):
    data[col] = data[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
data.head(3)

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f609,f610,f611,f612,f613,f614,f615,f616,f617,class
0,-0.4394,-0.0930,0.1718,0.4620,0.6226,0.4704,0.3578,0.0478,-0.1184,-0.2310,...,0.4102,0.2052,0.3846,0.3590,0.5898,0.3334,0.6410,0.5898,-0.4872,1
1,-0.4348,-0.1198,0.2474,0.4036,0.5026,0.6328,0.4948,0.0338,-0.0520,-0.1302,...,0.0000,0.2954,0.2046,0.4772,0.0454,0.2046,0.4318,0.4546,-0.0910,1
2,-0.2330,0.2124,0.5014,0.5222,-0.3422,-0.5840,-0.7168,-0.6342,-0.8614,-0.8318,...,-0.1112,-0.0476,-0.1746,0.0318,-0.0476,0.1112,0.2540,0.1588,-0.4762,2


In [97]:
data.dtypes.unique(), data["class"].value_counts(), data.isna().sum().sum()

(array([dtype('float64'), dtype('O')], dtype=object),
 class
 1    300
 2    300
 Name: count, dtype: int64,
 np.int64(0))

In [98]:
data.rename(columns={"class": "label"}, inplace=True)
data["label"] = data["label"].astype(int)
data.loc[data["label"] == 2, "label"] = 0
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,608,609,610,611,612,613,614,615,616,label
0,-0.4394,-0.0930,0.1718,0.4620,0.6226,0.4704,0.3578,0.0478,-0.1184,-0.2310,...,0.4102,0.2052,0.3846,0.3590,0.5898,0.3334,0.6410,0.5898,-0.4872,1
1,-0.4348,-0.1198,0.2474,0.4036,0.5026,0.6328,0.4948,0.0338,-0.0520,-0.1302,...,0.0000,0.2954,0.2046,0.4772,0.0454,0.2046,0.4318,0.4546,-0.0910,1
2,-0.2330,0.2124,0.5014,0.5222,-0.3422,-0.5840,-0.7168,-0.6342,-0.8614,-0.8318,...,-0.1112,-0.0476,-0.1746,0.0318,-0.0476,0.1112,0.2540,0.1588,-0.4762,0


In [99]:
data["label"].value_counts()

label
1    300
0    300
Name: count, dtype: int64

In [100]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,609,610,611,612,613,614,615,616,label,subset
0,-0.2558,0.2786,0.9270,0.9634,0.0958,-0.1278,-0.4794,-0.4658,-0.6028,-0.4156,...,0.7038,0.7284,0.1852,0.0864,0.4320,0.5062,-0.0124,-0.3580,0,train
1,-0.4236,0.1702,0.1752,0.5652,0.5378,0.7292,0.6448,0.2820,0.1354,0.1652,...,0.3146,0.3034,0.3596,0.1574,0.0562,-0.1460,-0.1012,-0.1910,1,train
2,-0.7572,-0.1230,-0.0622,0.5448,0.5600,0.4628,0.0956,-0.0622,-0.0166,-0.0166,...,-0.0230,0.0230,-0.0992,-0.0230,-0.2520,-0.2366,-0.2824,-0.3740,1,train
3,-0.4426,0.0366,0.3080,0.5502,0.7042,0.6968,0.1980,0.1100,-0.0440,-0.0268,...,0.3056,0.2084,0.5556,0.4584,0.4028,0.3056,-0.0278,-0.3056,1,train
4,-0.0450,0.6006,0.5106,0.6816,0.3934,-0.0360,-0.4384,-0.5166,-0.3604,-0.4624,...,0.0112,0.3484,0.3258,0.1460,0.2360,0.2584,-0.4382,-0.8876,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,-0.3408,0.0930,0.4002,0.5540,0.3408,-0.0210,-0.2590,-0.2516,-0.1796,-0.2366,...,0.4286,0.4966,0.7414,0.7414,0.5918,0.4966,0.3062,-0.0612,0,test
596,0.2080,1.0000,0.9758,0.8424,0.7374,0.0990,-0.0506,-0.2646,-0.4910,-0.4990,...,0.3334,0.0270,0.0090,-0.0090,-0.4234,-0.2252,-0.0630,-0.5316,0,test
597,-0.3458,0.0790,0.4118,0.5294,0.9540,0.9340,0.3888,0.2252,0.1850,0.0302,...,0.3846,0.5384,0.5164,0.4726,0.7362,0.6484,-0.0110,-0.4726,1,test
598,-0.1186,0.6216,1.0000,0.6050,-0.0978,-0.3596,-0.6882,-0.9210,-0.8586,-0.8712,...,0.1290,0.1452,-0.0968,0.0806,0.3388,0.2580,-0.1452,-0.5968,0,test


In [101]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.5
 1    0.5
 Name: proportion, dtype: float64,
 subset
 train    0.7
 test     0.3
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.5
         1        0.5
 train   0        0.5
         1        0.5
 Name: proportion, dtype: float64,
 np.int64(0))

In [102]:
data.to_parquet("../Benchmark/binary_isolet.parquet", index=False)

# SurvBoard-TCGA-BLCA-complete

In [134]:
with open("../Datasets/SurvBoard-TCGA-BLCA-complete/dataset_", "r", encoding="utf-8") as f:
    decoder = arff.ArffDecoder()
    obj = decoder.decode(f.read(), encode_nominal=True)

# Extract metadata
attributes = obj["attributes"]
data = obj["data"]

# Build DataFrame
columns = [attr[0] for attr in attributes]
data = pd.DataFrame(data, columns=columns)

print(data.shape)
data.head()

(50, 88519)


,patient_id,OS,OS_days,clinical_age_at_initial_pathologic_diagnosis,clinical_gender,clinical_race,clinical_ajcc_pathologic_tumor_stage,clinical_clinical_stage,clinical_histological_type,gex_?|100130426,...,rppa_X1433BETA,rppa_X1433ZETA,rppa_ACVRL1,rppa_DIRAS3,rppa_ANNEXIN1,rppa_PREX1,rppa_ERCC1,rppa_MSH2,rppa_MSH6,rppa_SMAC
0,TCGA-XF-AAN8,1,118,74,FEMALE,WHITE,Stage III,MISSING,Muscle invasive urothelial carcinoma (pT2 or a...,0.0,...,-0.576309,1.288440,-0.444005,-3.043338,1.053015,0.298932,-0.680792,-0.726141,-0.035625,0.153743
1,TCGA-CF-A5UA,0,365,67,MALE,ASIAN,Stage II,MISSING,Muscle invasive urothelial carcinoma (pT2 or a...,0.0,...,-0.762677,1.114102,-0.625459,-3.049802,-0.173575,-1.433333,-0.755080,-0.334523,0.137149,-0.275644
2,TCGA-BT-A0S7,1,200,75,MALE,WHITE,Stage III,MISSING,Muscle invasive urothelial carcinoma (pT2 or a...,0.0,...,-0.346402,0.965701,-0.449734,-2.765036,1.212702,-0.661940,-0.352659,-0.462832,0.207113,-0.341916
3,TCGA-GC-A3YS,0,758,61,MALE,WHITE,Stage IV,MISSING,Muscle invasive urothelial carcinoma (pT2 or a...,0.0,...,-0.581818,1.158213,-0.444801,-2.859959,1.143506,-1.013154,-0.406327,-0.636726,0.564707,-0.318785
4,TCGA-YC-A89H,0,573,78,FEMALE,WHITE,Stage II,MISSING,Muscle invasive urothelial carcinoma (pT2 or a...,0.0,...,-0.742976,2.006911,-0.735496,-3.275659,1.354597,-0.896192,-0.817930,-1.112323,-0.259771,0.283530


In [135]:
data.isna().sum().sum(), data.dtypes.unique()

(np.int64(0),
 array([dtype('O'), dtype('int64'), dtype('float64')], dtype=object))

In [136]:
data["patient_id"].nunique(), data["OS_days"].nunique(), data.shape

(50, 50, (50, 88519))

In [137]:
data.drop(columns=["patient_id", "OS_days"], inplace=True)
data.shape

(50, 88517)

In [138]:
categorical_cols = data.select_dtypes(include=["object", "category"]).columns.tolist()
for col in categorical_cols:
    print(f"{col}: {data[col].nunique()}")
    if data[col].nunique() == 1:
        data.drop(columns=[col], inplace=True)
data.shape

clinical_gender: 2
clinical_race: 4
clinical_ajcc_pathologic_tumor_stage: 3
clinical_clinical_stage: 1
clinical_histological_type: 1


(50, 88515)

In [ ]:
categorical_cols = data.select_dtypes(include=["object", "category"]).columns.tolist()
print(categorical_cols)
data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)
data.dtypes.unique(), data.shape

['clinical_gender', 'clinical_race', 'clinical_ajcc_pathologic_tumor_stage']


array([dtype('int64'), dtype('float64'), dtype('bool')], dtype=object)

In [140]:
boolean_cols = data.select_dtypes(include=["bool"]).columns.tolist()
for col in boolean_cols:
    data[col] = data[col].astype(int)
data.dtypes.unique()

array([dtype('int64'), dtype('float64')], dtype=object)

In [142]:
data["OS"].value_counts()

OS
0    30
1    20
Name: count, dtype: int64

In [144]:
data.rename(columns={"OS": "label"}, inplace=True)
data["label"].value_counts()

label
0    30
1    20
Name: count, dtype: int64

In [145]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,88508,88509,88510,88511,88512,88513,88514,88515,88516,label
0,1,74,0.0,2.523135,0.000000,6.626947,9.949322,0,7.997242,0.0000,...,-0.680792,-0.726141,-0.035625,0.153743,0,0,0,1,1,0
1,0,67,0.0,1.877352,3.928342,7.179869,10.464576,0,7.889248,1.8012,...,-0.755080,-0.334523,0.137149,-0.275644,1,0,0,0,0,0
2,1,75,0.0,0.000000,2.266337,8.319013,10.179337,0,6.932971,0.0000,...,-0.352659,-0.462832,0.207113,-0.341916,1,0,0,1,1,0


In [147]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,88509,88510,88511,88512,88513,88514,88515,88516,label,subset
0,1,73,0.000000,1.504570,2.559883,6.678945,10.524405,0,6.154334,0.000000,...,-0.556855,0.002390,0.306899,1,0,0,0,0,0,train
1,0,64,0.000000,3.941350,4.522206,7.708104,9.441216,0,8.193683,0.000000,...,-2.625485,-0.856469,1.284975,0,0,0,1,0,0,train
2,0,55,0.654161,2.876841,4.272374,6.703793,10.303518,0,9.113922,2.482797,...,-0.846522,-0.051360,0.752218,1,0,0,0,0,0,train


In [148]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64'), dtype('float64')], dtype=object),
 label
 0    0.76
 1    0.24
 Name: proportion, dtype: float64,
 subset
 train    0.7
 test     0.3
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.733333
         1        0.266667
 train   0        0.771429
         1        0.228571
 Name: proportion, dtype: float64,
 np.int64(0))

In [149]:
data.to_parquet("../Benchmark/suvboard_tcga_blca_complete.parquet", index=False)